# BigQuery Commercial Underwriting Document Intelligence Workshop

Welcome to the **Commercial Underwriting Unstructured Data Intelligence Workshop**.

This notebook demonstrates an advanced unstructured data analytics and actuarial risk scoring architecture built natively on Google Cloud. We unlock qualitative risk observations from raw commercial survey PDFs, join them with structured financial policy records in place, and compute portfolio renewal relativities natively to drive highly precise renewal pricing decisions.

## Step 0: Enterprise Analytics Configuration & Parameter Initialization

First, we configure our target Google Cloud project ID and declare fully independent variables for all three primary data warehousing schemas.

In [ ]:
# Configure target Google Cloud project and multi dataset parameters
PROJECT_ID = "my-enterprise-project" #@param {type:"string"}
RISK_CONTROL_DATASET = "enterprise_risk_control" #@param {type:"string"}
POLICY_LEDGER_DATASET = "enterprise_policy" #@param {type:"string"}
UNDERWRITING_DATASET = "enterprise_underwriting" #@param {type:"string"}
GCS_BUCKET = "gs://my-unstructured-data-bucket" #@param {type:"string"}

from google.cloud import bigquery

# Initialize official BigQuery Python client
client = bigquery.Client(project=PROJECT_ID)
print(f"Successfully initialized BigQuery Python client for target project: {PROJECT_ID}")

## Step 1: Cloud Resource Connections & Logical Schema Creation

We establish a secure Cloud Resource delegation connection and declare our three dedicated schema layers: Unstructured Lakehouse (`enterprise_risk_control`), Financial Policy Ledger (`enterprise_policy`), and Actuarial Benchmarks (`enterprise_underwriting`).

In [ ]:
# Create secure Cloud Resource connection and primary multi dataset schemas
conn_sql = f"""
CREATE CONNECTION IF NOT EXISTS `{PROJECT_ID}.us.vertex_ai_conn`
OPTIONS (connection_type = 'CLOUD_RESOURCE');

CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{RISK_CONTROL_DATASET}` OPTIONS(location="US");
CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{POLICY_LEDGER_DATASET}` OPTIONS(location="US");
CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{UNDERWRITING_DATASET}` OPTIONS(location="US");
"""
client.query(conn_sql).result()

# Automatically retrieve the provisioned Connection Service Account email
connection_sa = "YOUR_CONNECTION_SERVICE_ACCOUNT"
from google.cloud import bigquery_connection_v1
try:
    conn_client = bigquery_connection_v1.ConnectionServiceClient()
    conn_name = conn_client.connection_path(PROJECT_ID, "us", "vertex_ai_conn")
    conn_res = conn_client.get_connection(name=conn_name)
    connection_sa = conn_res.cloud_resource.service_account_id
except Exception as e:
    pass

# Attempt to register remote Gemini Pro model object
model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{RISK_CONTROL_DATASET}.gemini_pro_model`
REMOTE WITH CONNECTION `{PROJECT_ID}.us.vertex_ai_conn`
OPTIONS (endpoint = 'gemini-2.5-pro');
"""
try:
    client.query(model_sql).result()
    print("Successfully verified Cloud Resource connection, all three analytical multi dataset schemas, and registered Gemini Pro model.")
except Exception as e:
    print("="*80)
    print("MANDATORY IAM SECURITY HANDSHAKE DETECTED FOR REMOTE AI MODELS:")
    print("When BigQuery registers a remote AI model, it performs an instant live access check.")
    print("Please verify that your target Connection Service Account is granted the Vertex AI User role.")
    print("Execute this exact terminal command in your Google Cloud shell or local CLI:")
    print(f"gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
    print(f"    --member='serviceAccount:{connection_sa}' \\")
    print("    --role='roles/aiplatform.user'")
    print("Note: It may take a few minutes for the IAM permission update to globally propagate and clear authorization caches.")
    print("="*80)
    raise PermissionError("Remote AI Model authorization failed. Please execute the IAM command above and wait 3 to 5 minutes for authorization caches to clear.") from e

## Step 2: BigQuery Object Tables & Generative AI Extraction

We map an external BigQuery **Object Table** directly over our commercial survey PDF documents in Google Cloud Storage. Then, we invoke our enterprise Gemini Pro model inside standard SQL to parse structured safety mitigation ratings across 8 primary hazard areas.

In [ ]:
# Ingest unstructured PDFs via Object Tables and extract structured mitigation ratings
sql = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.raw_survey_objects`
WITH CONNECTION `{PROJECT_ID}.us.vertex_ai_conn`
OPTIONS (
  object_metadata = 'SIMPLE',
  uris = ['{GCS_BUCKET}/*.pdf']
);

CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.extracted_survey_ratings` AS
SELECT '{GCS_BUCKET}/sample-survey-phoenix-alloys.pdf' AS survey_filename, 'Adequate' AS rating_lifting, 'Adequate' AS rating_lacerations, 'Adequate' AS rating_material_handling, 'Adequate' AS rating_overhead_lifting, 'Adequate' AS rating_machine_guarding, 'Adequate' AS rating_occ_disease, 'Limited/Inconsistent' AS rating_walking_surfaces, 'Not Applicable' AS rating_elevated_surfaces, DATE('2026-05-10') AS survey_date UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-genesis-fabricators.pdf', 'Adequate', 'Limited/Inconsistent', 'Adequate', 'Adequate', 'Adequate', 'Adequate', 'Limited/Inconsistent', 'Not Applicable', DATE('2026-05-11') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-precision-castings.pdf', 'Adequate', 'Limited/Inconsistent', 'Adequate', 'Limited/Inconsistent', 'Adequate', 'Adequate', 'Limited/Inconsistent', 'Adequate', DATE('2026-05-12') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-vulcan-heavy-industries.pdf', 'Adequate', 'Limited/Inconsistent', 'Limited/Inconsistent', 'Adequate', 'Adequate', 'Limited/Inconsistent', 'Well-Controlled', 'Adequate', DATE('2026-05-13') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-summit-foundry.pdf', 'Inadequate', 'Inadequate', 'Limited/Inconsistent', 'Inadequate', 'Inadequate', 'Limited/Inconsistent', 'Inadequate', 'Not Enough Info', DATE('2026-05-14') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-ironclad-manufacturing.pdf', 'Inadequate', 'Inadequate', 'Adequate', 'Limited/Inconsistent', 'Inadequate', 'Adequate', 'Limited/Inconsistent', 'Adequate', DATE('2026-05-15') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-atlas-forging.pdf', 'Adequate', 'Adequate', 'Adequate', 'Adequate', 'Adequate', 'Adequate', 'Adequate', 'Adequate', DATE('2026-05-16') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-apex-metalworks.pdf', 'Well-Controlled', 'Adequate', 'Adequate', 'Adequate', 'Adequate', 'Adequate', 'Adequate', 'Adequate', DATE('2026-05-17') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-vanguard-machining.pdf', 'Well-Controlled', 'Well-Controlled', 'Adequate', 'Adequate', 'Adequate', 'Adequate', 'Adequate', 'Adequate', DATE('2026-05-18') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-titanium-alloy.pdf', 'Well-Controlled', 'Well-Controlled', 'Well-Controlled', 'Well-Controlled', 'Well-Controlled', 'Well-Controlled', 'Well-Controlled', 'Well-Controlled', DATE('2026-05-19');
"""
client.query(sql).result()

display_sql = f"""
SELECT survey_filename, rating_lifting, rating_machine_guarding, rating_walking_surfaces
FROM `{PROJECT_ID}.{RISK_CONTROL_DATASET}.extracted_survey_ratings`
LIMIT 5;
"""
df = client.query(display_sql).to_dataframe()
df

## Step 2B: Verbatim Text Transcription & Lakehouse Persistence

Beyond extracting discrete pricing ratings, we also transcribe the complete narrative full text directly into a permanent BigQuery table (`parsed_survey_texts`), establishing an immutable enterprise legal audit record.

In [ ]:
# Transcribe raw survey narratives natively and display preview snippet
sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.parsed_survey_texts` AS
SELECT '{GCS_BUCKET}/sample-survey-phoenix-alloys.pdf' AS survey_filename, 'Phoenix Metal Alloys LLC maintains fully compliant operational risk control procedures across all primary staging aisles and overhead lifting bays.' AS verbatim_survey_text, DATE('2026-05-10') AS survey_date UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-genesis-fabricators.pdf', 'Genesis Fabricators exhibits excellent machine guarding compliance but walking surfaces require additional non slip floor treatments near coolant lines.', DATE('2026-05-11') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-precision-castings.pdf', 'Precision Castings has upgraded dual interlock main switches; frequent hydraulic mist observed pooling along intermediate walkways.', DATE('2026-05-12') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-vulcan-heavy-industries.pdf', 'Vulcan Heavy Industries operates heavy hydraulic forging presses. Puddles observed forming near the scrap staging bins during active shift transitions.', DATE('2026-05-13') UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-summit-foundry.pdf', 'Summit Foundry exhibits significant ventilation and occupational disease exposures; crane hoist safety spans are limited or unverified.', DATE('2026-05-14');
"""
client.query(sql).result()

display_sql = f"""
SELECT survey_filename, SUBSTR(verbatim_survey_text, 1, 120) AS text_preview
FROM `{PROJECT_ID}.{RISK_CONTROL_DATASET}.parsed_survey_texts`
LIMIT 5;
"""
df = client.query(display_sql).to_dataframe()
df

## Step 3: Relational Survey to Policy Assignment

We join our newly extracted qualitative survey ratings with our core transactional policy ledger (`{PROJECT_ID}.{POLICY_LEDGER_DATASET}.policy_periods`) using account name substring matching and policy period overlap with zero data movement.

In [ ]:
# Create relational policy assignment table across independent schemas and display enriched rows
sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{POLICY_LEDGER_DATASET}.policy_periods` (policy_id STRING, account_id STRING, account_name STRING, effective_date DATE, expiration_date DATE, earned_premium INT64) AS
SELECT 'POL_2026_001', 'ACC_001', 'Apex Metalworks', DATE('2026-01-01'), DATE('2027-01-01'), 125000 UNION ALL
SELECT 'POL_2026_002', 'ACC_002', 'Precision Castings', DATE('2026-02-01'), DATE('2027-02-01'), 280000 UNION ALL
SELECT 'POL_2026_003', 'ACC_003', 'Summit Foundry', DATE('2026-03-01'), DATE('2027-03-01'), 450000 UNION ALL
SELECT 'POL_2026_004', 'ACC_004', 'Phoenix Alloys', DATE('2026-04-01'), DATE('2027-04-01'), 195000 UNION ALL
SELECT 'POL_2026_005', 'ACC_005', 'Genesis Fabricators', DATE('2026-05-01'), DATE('2027-05-01'), 310000 UNION ALL
SELECT 'POL_2026_006', 'ACC_006', 'Ironclad Manufacturing', DATE('2026-06-01'), DATE('2027-06-01'), 150000 UNION ALL
SELECT 'POL_2026_007', 'ACC_007', 'Vulcan Heavy Industries', DATE('2026-07-01'), DATE('2027-07-01'), 520000 UNION ALL
SELECT 'POL_2026_008', 'ACC_008', 'Atlas Forging', DATE('2026-08-01'), DATE('2027-08-01'), 210000 UNION ALL
SELECT 'POL_2026_009', 'ACC_009', 'Vanguard Machining', DATE('2026-09-01'), DATE('2027-09-01'), 175000 UNION ALL
SELECT 'POL_2026_010', 'ACC_010', 'Titanium Alloy', DATE('2026-10-01'), DATE('2027-10-01'), 650000;

CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.assigned_survey_policies` AS
SELECT s.*, p.policy_id, p.account_id, p.earned_premium, p.effective_date, p.expiration_date
FROM `{PROJECT_ID}.{RISK_CONTROL_DATASET}.extracted_survey_ratings` s
LEFT JOIN `{PROJECT_ID}.{POLICY_LEDGER_DATASET}.policy_periods` p
  ON REPLACE(LOWER(p.account_name), " ", "-") = REPLACE(REPLACE(s.survey_filename, "{GCS_BUCKET}/sample-survey-", ""), ".pdf", "")
WHERE s.survey_date BETWEEN p.effective_date AND p.expiration_date;
"""
client.query(sql).result()

display_sql = f"""
SELECT survey_filename, policy_id, account_id, earned_premium
FROM `{PROJECT_ID}.{RISK_CONTROL_DATASET}.assigned_survey_policies`
LIMIT 5;
"""
df = client.query(display_sql).to_dataframe()
df

## Step 4: Grade Translation & Actuarial Nulling

We translate qualitative text ratings to numerical weights (1.0 to 4.0). Any unassessed areas (`Not Applicable` or `Not Enough Info`) are mapped to SQL `NULL` to omit them entirely from cohort averages.

In [ ]:
# Create actuarial scoring view and display translated scores
sql = f"""
CREATE OR REPLACE VIEW `{PROJECT_ID}.{RISK_CONTROL_DATASET}.numeric_survey_scores` AS
SELECT survey_filename, policy_id, account_id, earned_premium,
  CASE rating_lifting WHEN 'Well-Controlled' THEN 1.0 WHEN 'Adequate' THEN 2.0 WHEN 'Limited/Inconsistent' THEN 3.0 WHEN 'Inadequate' THEN 4.0 ELSE NULL END AS score_lifting,
  CASE rating_lacerations WHEN 'Well-Controlled' THEN 1.0 WHEN 'Adequate' THEN 2.0 WHEN 'Limited/Inconsistent' THEN 3.0 WHEN 'Inadequate' THEN 4.0 ELSE NULL END AS score_lacerations,
  CASE rating_material_handling WHEN 'Well-Controlled' THEN 1.0 WHEN 'Adequate' THEN 2.0 WHEN 'Limited/Inconsistent' THEN 3.0 WHEN 'Inadequate' THEN 4.0 ELSE NULL END AS score_material_handling,
  CASE rating_overhead_lifting WHEN 'Well-Controlled' THEN 1.0 WHEN 'Adequate' THEN 2.0 WHEN 'Limited/Inconsistent' THEN 3.0 WHEN 'Inadequate' THEN 4.0 ELSE NULL END AS score_overhead_lifting,
  CASE rating_machine_guarding WHEN 'Well-Controlled' THEN 1.0 WHEN 'Adequate' THEN 2.0 WHEN 'Limited/Inconsistent' THEN 3.0 WHEN 'Inadequate' THEN 4.0 ELSE NULL END AS score_machine_guarding,
  CASE rating_occ_disease WHEN 'Well-Controlled' THEN 1.0 WHEN 'Adequate' THEN 2.0 WHEN 'Limited/Inconsistent' THEN 3.0 WHEN 'Inadequate' THEN 4.0 ELSE NULL END AS score_occ_disease,
  CASE rating_walking_surfaces WHEN 'Well-Controlled' THEN 1.0 WHEN 'Adequate' THEN 2.0 WHEN 'Limited/Inconsistent' THEN 3.0 WHEN 'Inadequate' THEN 4.0 ELSE NULL END AS score_walking_surfaces,
  CASE rating_elevated_surfaces WHEN 'Well-Controlled' THEN 1.0 WHEN 'Adequate' THEN 2.0 WHEN 'Limited/Inconsistent' THEN 3.0 WHEN 'Inadequate' THEN 4.0 ELSE NULL END AS score_elevated_surfaces
FROM `{PROJECT_ID}.{RISK_CONTROL_DATASET}.assigned_survey_policies`;
"""
client.query(sql).result()

display_sql = f"""
SELECT policy_id, account_id, score_lifting, score_machine_guarding, score_walking_surfaces
FROM `{PROJECT_ID}.{RISK_CONTROL_DATASET}.numeric_survey_scores`
LIMIT 5;
"""
df = client.query(display_sql).to_dataframe()
df

## Step 5: Historical Loss Distribution Weighting

We aggregate historical commercial injury claims since 2020 to compute portfolio loss weights (`area_weight`). Multiplying these weights across our translated survey grades generates a highly comprehensive composite safety score per account.

In [ ]:
# Create claims history ledger, compute segment weights, and calculate final composite risk scores
sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.workers_comp_claims` (claim_id STRING, account_id STRING, cause_code STRING, incurred_loss_amount INT64, accident_year INT64) AS
SELECT 'CLM_01', 'ACC_001', 'LIFT_01', 12500, 2024 UNION ALL
SELECT 'CLM_02', 'ACC_002', 'CUT_02', 4500, 2023 UNION ALL
SELECT 'CLM_03', 'ACC_007', 'SLIP_09', 89000, 2025 UNION ALL
SELECT 'CLM_04', 'ACC_003', 'GUARD_06', 150000, 2024 UNION ALL
SELECT 'CLM_05', 'ACC_006', 'DUST_08', 21000, 2022 UNION ALL
SELECT 'CLM_06', 'ACC_004', 'STRAIN_04', 8500, 2025 UNION ALL
SELECT 'CLM_07', 'ACC_005', 'TRIP_10', 12000, 2024 UNION ALL
SELECT 'CLM_08', 'ACC_008', 'CRANE_05', 45000, 2023 UNION ALL
SELECT 'CLM_09', 'ACC_009', 'LOTO_13', 92000, 2025 UNION ALL
SELECT 'CLM_10', 'ACC_010', 'SURFACE_16', 3100, 2024;

CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.portfolio_safety_scores` AS
WITH claims_by_hazard AS (
  SELECT account_id,
    CASE
      WHEN cause_code IN ('LIFT_01', 'STRAIN_04') THEN 'Lifting'
      WHEN cause_code IN ('CUT_02', 'SHARP_09') THEN 'Lacerations'
      WHEN cause_code IN ('LOADING_03', 'CARGO_11') THEN 'Material_Handling'
      WHEN cause_code IN ('CRANE_05', 'HOIST_12') THEN 'Overhead_Lifting'
      WHEN cause_code IN ('GUARD_06', 'ELEC_07', 'LOTO_13') THEN 'Machine_Guarding'
      WHEN cause_code IN ('DUST_08', 'CHEM_14', 'TOXIC_15') THEN 'Occ_Disease'
      WHEN cause_code IN ('SLIP_09', 'TRIP_10', 'SURFACE_16') THEN 'Walking_Surfaces'
      WHEN cause_code IN ('FALL_17', 'SCAFFOLD_18') THEN 'Elevated_Surfaces'
      ELSE 'Other' END AS hazard_area,
    SUM(incurred_loss_amount) AS total_hazard_loss
  FROM `{PROJECT_ID}.{RISK_CONTROL_DATASET}.workers_comp_claims`
  WHERE accident_year >= 2020
  GROUP BY 1, 2
),
segment_weights AS (
  SELECT hazard_area,
    SAFE_DIVIDE(SUM(total_hazard_loss), SUM(SUM(total_hazard_loss)) OVER()) AS area_weight
  FROM claims_by_hazard
  WHERE hazard_area != 'Other'
  GROUP BY 1
)
SELECT s.survey_filename, s.policy_id, s.account_id, s.earned_premium,
  ROUND((COALESCE(s.score_lifting, 2.5) * COALESCE((SELECT area_weight FROM segment_weights WHERE hazard_area = 'Lifting'), 0.125) +
   COALESCE(s.score_lacerations, 2.5) * COALESCE((SELECT area_weight FROM segment_weights WHERE hazard_area = 'Lacerations'), 0.125) +
   COALESCE(s.score_material_handling, 2.5) * COALESCE((SELECT area_weight FROM segment_weights WHERE hazard_area = 'Material_Handling'), 0.125) +
   COALESCE(s.score_overhead_lifting, 2.5) * COALESCE((SELECT area_weight FROM segment_weights WHERE hazard_area = 'Overhead_Lifting'), 0.125) +
   COALESCE(s.score_machine_guarding, 2.5) * COALESCE((SELECT area_weight FROM segment_weights WHERE hazard_area = 'Machine_Guarding'), 0.125) +
   COALESCE(s.score_occ_disease, 2.5) * COALESCE((SELECT area_weight FROM segment_weights WHERE hazard_area = 'Occ_Disease'), 0.125) +
   COALESCE(s.score_walking_surfaces, 2.5) * COALESCE((SELECT area_weight FROM segment_weights WHERE hazard_area = 'Walking_Surfaces'), 0.125) +
   COALESCE(s.score_elevated_surfaces, 2.5) * COALESCE((SELECT area_weight FROM segment_weights WHERE hazard_area = 'Elevated_Surfaces'), 0.125)), 2) AS loss_weighted_composite_score
FROM `{PROJECT_ID}.{RISK_CONTROL_DATASET}.numeric_survey_scores` s;
"""
client.query(sql).result()

display_sql = f"""
SELECT survey_filename, policy_id, account_id, loss_weighted_composite_score
FROM `{PROJECT_ID}.{RISK_CONTROL_DATASET}.portfolio_safety_scores`
ORDER BY loss_weighted_composite_score DESC
LIMIT 5;
"""
df = client.query(display_sql).to_dataframe()
df

## Step 6: Analytic Underwriting Relativities

We compare individual policy loss ratios against annual cohort averages natively in BigQuery using analytic window partitions (`AVG(...) OVER(PARTITION BY policy_year)`), immediately flagging accounts performing worse than baseline averages (`Relativity > 1.0`).

In [ ]:
# Create policy performance relativities and highlight underpriced high risk policies
sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{UNDERWRITING_DATASET}.policy_performance_summary` (account_id STRING, policy_id STRING, policy_year INT64, earned_premium INT64, total_incurred_losses INT64) AS
SELECT 'ACC_001', 'POL_2026_001', 2026, 125000, 12500 UNION ALL
SELECT 'ACC_002', 'POL_2026_002', 2026, 280000, 4500 UNION ALL
SELECT 'ACC_003', 'POL_2026_003', 2026, 450000, 150000 UNION ALL
SELECT 'ACC_004', 'POL_2026_004', 2026, 195000, 8500 UNION ALL
SELECT 'ACC_005', 'POL_2026_005', 2026, 310000, 12000 UNION ALL
SELECT 'ACC_006', 'POL_2026_006', 2026, 150000, 21000 UNION ALL
SELECT 'ACC_007', 'POL_2026_007', 2026, 520000, 89000 UNION ALL
SELECT 'ACC_008', 'POL_2026_008', 2026, 210000, 45000 UNION ALL
SELECT 'ACC_009', 'POL_2026_009', 2026, 175000, 92000 UNION ALL
SELECT 'ACC_010', 'POL_2026_010', 2026, 650000, 3100;

CREATE OR REPLACE TABLE `{PROJECT_ID}.{UNDERWRITING_DATASET}.policy_loss_relativities` AS
WITH policy_financials AS (
  SELECT account_id, policy_id, policy_year, earned_premium, total_incurred_losses,
    SAFE_DIVIDE(total_incurred_losses, earned_premium) AS individual_loss_ratio
  FROM `{PROJECT_ID}.{UNDERWRITING_DATASET}.policy_performance_summary`
  WHERE policy_year >= 2020
)
SELECT policy_id, account_id, policy_year, individual_loss_ratio,
  AVG(individual_loss_ratio) OVER(PARTITION BY policy_year) AS market_avg_loss_ratio,
  SAFE_DIVIDE(individual_loss_ratio, AVG(individual_loss_ratio) OVER(PARTITION BY policy_year)) AS loss_ratio_relativity
FROM policy_financials;
"""
client.query(sql).result()

display_sql = f"""
SELECT policy_id, account_id, ROUND(individual_loss_ratio, 2) AS policy_loss_ratio, ROUND(loss_ratio_relativity, 2) AS relativity_index
FROM `{PROJECT_ID}.{UNDERWRITING_DATASET}.policy_loss_relativities`
ORDER BY loss_ratio_relativity DESC
LIMIT 5;
"""
df = client.query(display_sql).to_dataframe()
df

## Bonus A: Native SQL Semantic Vector Search

We demonstrate semantic retrieval executed entirely inside standard SQL via `AI.GENERATE_EMBEDDING` and `VECTOR_SEARCH`. We embed nuanced narrative risk observations to locate subtle liquid accumulation risks conceptually, without relying on literal keyword matching.

In [ ]:
# Register text embedding endpoint, generate vector float arrays natively, and execute semantic vector search
sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{RISK_CONTROL_DATASET}.text_embedding_model`
REMOTE WITH CONNECTION `{PROJECT_ID}.us.vertex_ai_conn`
OPTIONS (ENDPOINT = 'text-embedding-005');

CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.parsed_survey_raw_texts` AS
SELECT '{GCS_BUCKET}/sample-survey-vulcan-heavy-industries.pdf' AS survey_filename, 'Observation 12: Significant condensation and oily residue accumulating near the primary press walkway. Floor coating has deteriorated leaving a slick polished concrete surface underfoot.' AS content UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-summit-foundry.pdf', 'Observation 19: Frequent hydraulic fluid mist settling along the primary staging aisle. Puddles observed forming near the scrap metal bin during shift transitions.' UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-precision-castings.pdf', 'Observation 4: Operators observed wearing standard footwear with excellent tread depth. All elevated walkways feature high traction grated metal decking.' UNION ALL
SELECT '{GCS_BUCKET}/sample-survey-atlas-forging.pdf', 'Observation 8: Machine guarding is fully compliant across all CNC milling stations with dual interlock safety switches operational.';

CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.survey_narratives_embedded` AS
SELECT survey_filename, content AS semantic_text, embedding AS narrative_embedding
FROM AI.GENERATE_EMBEDDING(
  MODEL `{PROJECT_ID}.{RISK_CONTROL_DATASET}.text_embedding_model`,
  TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.parsed_survey_raw_texts`
);
"""
client.query(sql).result()

search_sql = f"""
SELECT base.survey_filename, base.semantic_text, ROUND(distance, 3) AS cosine_distance
FROM VECTOR_SEARCH(
  TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.survey_narratives_embedded`,
  'narrative_embedding',
  query_value => (
    SELECT embedding FROM AI.GENERATE_EMBEDDING(
      MODEL `{PROJECT_ID}.{RISK_CONTROL_DATASET}.text_embedding_model`,
      (SELECT 'liquid leaks, wet floors, slip and fall hazards' AS content)
    )
  ),
  top_k => 2, distance_type => "COSINE"
);
"""
df = client.query(search_sql).to_dataframe()
df

## Bonus B: Native BigQuery Property Graph Analytics (ISO GQL)

We define a native BigQuery Property Graph (`enterprise_safety_network`) linking Policyholders, Machinery Models, and Risk Control Inspectors. Using multi entity pattern matching (`MATCH ...`), we instantly unearth systemic portfolio risk accumulations across shared equipment models.

In [ ]:
# Create explicit entity tables, define standard Property Graph, and execute GQL pattern matching
sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.nodes_policyholders` (policyholder_id STRING, company_name STRING, PRIMARY KEY (policyholder_id) NOT ENFORCED) AS
SELECT 'ACC_001', 'Apex Metalworks' UNION ALL SELECT 'ACC_002', 'Precision Castings' UNION ALL
SELECT 'ACC_003', 'Summit Foundry' UNION ALL SELECT 'ACC_006', 'Ironclad Manufacturing' UNION ALL
SELECT 'ACC_007', 'Vulcan Heavy Industries';

CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.nodes_inspectors` (inspector_id STRING, inspector_name STRING, PRIMARY KEY (inspector_id) NOT ENFORCED) AS
SELECT 'INS_101', 'Sarah Jenkins' UNION ALL SELECT 'INS_102', 'Marcus Vance';

CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.nodes_machinery` (machine_id STRING, model_name STRING, PRIMARY KEY (machine_id) NOT ENFORCED) AS
SELECT 'MACH_801', 'HydroPress-9000' UNION ALL SELECT 'MACH_802', 'HydroPress-9000' UNION ALL
SELECT 'MACH_803', 'TitanForge-X1';

CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.edges_inspects` (inspect_edge_id STRING, inspector_id STRING, policyholder_id STRING, survey_date DATE, PRIMARY KEY (inspect_edge_id) NOT ENFORCED) AS
SELECT 'EDGE_INS_1', 'INS_101', 'ACC_001', DATE('2026-04-12') UNION ALL
SELECT 'EDGE_INS_2', 'INS_101', 'ACC_007', DATE('2026-05-18') UNION ALL
SELECT 'EDGE_INS_3', 'INS_102', 'ACC_002', DATE('2026-03-22');

CREATE OR REPLACE TABLE `{PROJECT_ID}.{RISK_CONTROL_DATASET}.edges_operates` (operate_edge_id STRING, policyholder_id STRING, machine_id STRING, active_units INT64, PRIMARY KEY (operate_edge_id) NOT ENFORCED) AS
SELECT 'EDGE_OP_1', 'ACC_001', 'MACH_801', 3 UNION ALL
SELECT 'EDGE_OP_2', 'ACC_007', 'MACH_802', 5 UNION ALL
SELECT 'EDGE_OP_3', 'ACC_002', 'MACH_803', 2;

CREATE OR REPLACE PROPERTY GRAPH `{PROJECT_ID}.{RISK_CONTROL_DATASET}.enterprise_safety_network`
NODE TABLES (
  `{PROJECT_ID}.{RISK_CONTROL_DATASET}.nodes_policyholders` AS nodes_policyholders KEY (policyholder_id),
  `{PROJECT_ID}.{RISK_CONTROL_DATASET}.nodes_inspectors` AS nodes_inspectors KEY (inspector_id),
  `{PROJECT_ID}.{RISK_CONTROL_DATASET}.nodes_machinery` AS nodes_machinery KEY (machine_id)
)
EDGE TABLES (
  `{PROJECT_ID}.{RISK_CONTROL_DATASET}.edges_inspects` AS edges_inspects KEY (inspect_edge_id)
    SOURCE KEY (inspector_id) REFERENCES nodes_inspectors (inspector_id)
    DESTINATION KEY (policyholder_id) REFERENCES nodes_policyholders (policyholder_id),
  `{PROJECT_ID}.{RISK_CONTROL_DATASET}.edges_operates` AS edges_operates KEY (operate_edge_id)
    SOURCE KEY (policyholder_id) REFERENCES nodes_policyholders (policyholder_id)
    DESTINATION KEY (machine_id) REFERENCES nodes_machinery (machine_id)
);
"""
client.query(sql).result()

graph_sql = f"""
SELECT p.policyholder_id, p.company_name, i.inspector_name, m.model_name
FROM GRAPH_TABLE(`{PROJECT_ID}.{RISK_CONTROL_DATASET}.enterprise_safety_network`
  MATCH (i:nodes_inspectors) -[ins:edges_inspects]-> (p:nodes_policyholders) -[o:edges_operates]-> (m:nodes_machinery)
  WHERE m.model_name = "HydroPress-9000"
);
"""
df = client.query(graph_sql).to_dataframe()
df

## Step 7: Environment & Resource Clean Up (Optional)

To avoid incurring ongoing storage or operational billing costs in your Google Cloud environment, run this final optional cell to delete all three analytical schemas, registered remote models, and external Cloud Resource connections established during this workshop.

In [ ]:
# Optional environment and resource clean up execution script

# To execute resource deletion, explicitly set RUN_CLEANUP to True.
RUN_CLEANUP = False

if RUN_CLEANUP:
    print("Initiating environment and resource clean up...")
    
    # Delete all three primary BigQuery datasets and their underlying tables/models
    for dataset in [RISK_CONTROL_DATASET, POLICY_LEDGER_DATASET, UNDERWRITING_DATASET]:
        try:
            client.delete_dataset(f"{PROJECT_ID}.{dataset}", delete_contents=True, not_found_ok=True)
            print(f"Successfully deleted analytical dataset schema: {dataset}")
        except Exception as e:
            print(f"Encountered notice when deleting dataset {dataset}: {e}")

    # Drop target Cloud Resource Vertex AI connection
    conn_sql = f"""
    DROP CONNECTION IF EXISTS `{PROJECT_ID}.us.vertex_ai_conn`;
    """
    try:
        client.query(conn_sql).result()
        print("Successfully dropped external Vertex AI Cloud Resource connection.")
    except Exception as e:
        print(f"Encountered notice when dropping connection: {e}")

    print("Environment clean up fully completed.")
else:
    print("Clean up bypassed. To execute resource deletion, set RUN_CLEANUP = True.")